In [31]:
# import the required liberaries
import os
import asyncio
from tavily import TavilyClient
from dotenv import load_dotenv
from langchain.messages import SystemMessage,AIMessage,HumanMessage
from langchain.tools import tool
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from pydantic import BaseModel
from typing import List,Dict

load_dotenv()
client=TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["GEMINI_API_KEY"]=os.getenv("GEMINI_API_KEY")

In [ ]:
# define all the tools 
@tool
def getAllDeliveryPartners()-> List[str]:
    """
    Returns a list of available delivery partners for parcel shipping.

    These partners can be evaluated based on cost, speed, reliability,
    and service coverage to select the best options. 
    """
    return ["DHL","Delhivery","BlueDart","Ecom Express","XpressBees","FedEx","Shadowfax",'amazon logistics','ekart']

@tool
async def getinfo(allPartners:List[str])->List[Dict]:
    """
     Fetch detailed information about a list of delivery partners using web search.

    Args:
        allPartners: List of delivery partner names (strings)

    Returns:
        List of dictionaries containing structured partner information
    """
    partners_info=[]
    for p in allPartners:
        try:
            info= tavily_Websearch_Agent(p)
            partners_info.append({
                "partner":p,
                "information":info,
            })
        except Exception as e:
            partners_info.append({
                "partner":p,
                "information":e,
            })
    return partners_info


def tavily_Websearch_Agent(partnerName:str)->str:
    query=f"{partnerName} courier delivery speed reliability India coverage"
    results= client.search(query=query,max_results=3)
    text = ""

    for r in results["results"]:
        text += r["content"] + "\n"
    return text



In [42]:
system_prompt=""" 
You are an intelligent logistics decision-making agent.

Your job is to select the BEST 3 delivery partners for a given parcel request.

You MUST follow this process:

1. Call getAllDeliveryPartners to get all available partners
2. Call getinfo to gather information about them
3. Analyze based on:
   - Cost
   - Speed
   - Reliability
   - Coverage

4. Select exactly 3 partners

Rules:
- Only choose from the available partner list
- Do not hallucinate new partners
- Always use tools before deciding
- Be logical and concise

Return your response strictly in the required structured format.
"""

system_prompt_2=""" 
You are an intelligent logistics decision-making agent.

Your job is to select the BEST 3 delivery partners for a given parcel information.
Select exactly 3 partners

Rules:
- Only choose from the available partner list
- Do not hallucinate new partners
- use can use tools 
- Be logical and concise

Return your response strictly in the required structured format.
"""

In [43]:
# initilize the model & define the agent
model=ChatGroq(model="qwen/qwen3-32b")

class OutputSchema(BaseModel):
    top_partners:List[str]

agent=create_agent(
    model=model,
    tools=[getAllDeliveryPartners,getinfo],
    system_prompt=system_prompt_2,
    response_format= OutputSchema
)
# agent

In [45]:
res=await agent.ainvoke({"messages":HumanMessage("{'weight': 12.0, 'pickup_city': 'rohtak', 'delivery_city': 'jind', 'delivery_speed': 'standard', 'total_km': 200.0}")})
print(res)

{'messages': [HumanMessage(content="{'weight': 12.0, 'pickup_city': 'rohtak', 'delivery_city': 'jind', 'delivery_speed': 'standard', 'total_km': 200.0}", additional_kwargs={}, response_metadata={}, id='b374f3b9-724f-4856-bfcc-cc49ebfb9cac'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, let's tackle this problem step by step. The user needs the best 3 delivery partners for a parcel with specific details: weight 12kg, pickup in Rohtak, delivery to Jind, standard speed, 200 km. First, I need to get the list of available partners using the getAllDeliveryPartners function. Once I have that, I can use getinfo to fetch details on each. But since the user hasn't provided the actual partner names yet, maybe the first step is to call getAllDeliveryPartners to get the list. Wait, but the user's message doesn't include any partners. Hmm, maybe the system is designed so that the agent first retrieves all possible partners, then evaluates them based on the parcel's criteria. 

In [46]:
print(res["structured_response"])

top_partners=['Delhivery', 'BlueDart', 'XpressBees']


In [47]:
def debug_agent(res):
    for msg in res["messages"]:
        role = msg.__class__.__name__.replace("Message", "")
        print(f"\n[{role}]")

        if hasattr(msg, "content") and msg.content:
            print(msg.content)

        if hasattr(msg, "additional_kwargs") and msg.additional_kwargs:
            print("KWARGS:", msg.additional_kwargs)

debug_agent(res)


[Human]
{'weight': 12.0, 'pickup_city': 'rohtak', 'delivery_city': 'jind', 'delivery_speed': 'standard', 'total_km': 200.0}

[AI]
KWARGS: {'reasoning_content': "Okay, let's tackle this problem step by step. The user needs the best 3 delivery partners for a parcel with specific details: weight 12kg, pickup in Rohtak, delivery to Jind, standard speed, 200 km. First, I need to get the list of available partners using the getAllDeliveryPartners function. Once I have that, I can use getinfo to fetch details on each. But since the user hasn't provided the actual partner names yet, maybe the first step is to call getAllDeliveryPartners to get the list. Wait, but the user's message doesn't include any partners. Hmm, maybe the system is designed so that the agent first retrieves all possible partners, then evaluates them based on the parcel's criteria. So the initial step is to call getAllDeliveryPartners. Once I have that list, I can pass it to getinfo to gather detailed info like cost, speed